In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text('incremental_flag','o')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')
print(incremental_flag)

In [0]:
df_src = spark.sql(
    """
        select Date_ID from PARQUET.`abfss://silver@carsalesdata.dfs.core.windows.net/carsales`
    """
)
df_src.display()

In [0]:
if not  spark.catalog.tableExists('cars_catalog.gold.dim_date'):
    df_sink = spark.sql(
        '''
            SELECT 1 AS dim_date_key, Date_ID
            FROM PARQUET.`abfss://silver@carsalesdata.dfs.core.windows.net/carsales`
            WHERE 1=0
        '''
    )
else:
    df_sink = spark.sql(
        '''
            SELECT dim_date_key, Date_ID
            FROM cars_catalog.gold.dim_date
        '''
    )

In [0]:
df_sink.display()

In [0]:
df_filter = df_src.join(df_sink, df_src.Date_ID == df_sink.Date_ID, 'left').select(df_src.Date_ID,df_sink.dim_date_key)
df_filter = df_filter.dropDuplicates(["dim_date_key"])
df_filter.display()


In [0]:
df_filter_old = df_filter.filter(df_filter.dim_date_key.isNotNull())
df_filter_old.display()
df_filter_new = df_filter.filter(df_filter.dim_date_key.isNull()).select('Date_ID')
df_filter_new.display()

In [0]:
if incremental_flag=='0':
    max_value =0
else:
    if spark.catalog.tableExists('cars_catalog.gold.dim_date'):
        max_value = spark.sql("select max(dim_date_key) from cars_catalog.gold.dim_date").first()[0]
    else:
        max_value = 0
print(max_value)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

window = Window.orderBy(F.lit(1))

df_filter_new = df_filter_new.withColumn('dim_date_key', F.row_number().over(window) + max_value)
df_filter_new.display()

In [0]:
df_final = df_filter_new.union(df_filter_old)
df_final.display()

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists('cars_catalog.gold.dim_date'):
    delta_table = DeltaTable.forPath(spark,"abfss://gold@carsalesdata.dfs.core.windows.net/dim_date")
    delta_table.alias("t").merge(df_final.alias("s"),"t.dim_date_key = s.dim_date_key")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
else:
    df_final.write.format('delta')\
        .mode('overwrite')\
        .option("path",'abfss://gold@carsalesdata.dfs.core.windows.net/dim_date')\
        .saveAsTable("cars_catalog.gold.dim_date")


In [0]:
%sql
select * from cars_catalog.gold.dim_date;